# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aniqaatiq842-commits/Flyrank-ML-INTERNSHIP/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
### Rule

I will prioritize pages for content refresh when they show stronger evidence of
being stale while still having meaningful search opportunity.

The baseline uses two observable signals:

1. Content age — used as the staleness signal.
2. Search visibility — used as the opportunity signal.

The score combines the two signals into one transparent priority score.
Higher scores receive higher refresh priority.

### Reason code

- `STALE_WITH_SEARCH_OPPORTUNITY` — the page is relatively old and still has meaningful search visibility.

### Action

- `REFRESH`

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("refresh_queue_sample.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

print("\nColumns:")
for col in df.columns:
    print("-", col)

display(df.head())

Dataset shape: (200, 28)

Columns:
- final_rank
- content_id
- client_id
- final_refresh_score
- best_model_name
- best_model_probability
- baseline_refresh_score
- confidence
- suggested_action
- final_reason_codes
- is_declining_label
- impressions_90d
- clicks_90d
- sessions_90d
- avg_position
- ctr
- content_age_days
- days_since_last_update
- word_count
- trend_direction
- competition_level
- content_type
- main_intent
- age_tier
- freshness_tier
- word_count_tier
- impression_tier
- position_tier


,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,is_declining_label,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,1,content_1f080331fa2b,client_3fdba35f04,81.636697,random_forest,0.782079,0.844481,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|...,1,12834,6,66,6.8,0.05,165,104,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1
1,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,random_forest,0.788105,0.825477,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate,1,8064,6,23,3.8,0.07,139,104,1457.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
2,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,random_forest,0.847372,0.695884,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate,1,2498,0,9,10.1,0.00,165,104,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
3,4,content_72e800a9c214,client_3fdba35f04,81.034960,random_forest,0.774371,0.842545,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate,1,13790,16,27,8.2,0.12,139,104,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1
4,5,content_e04eb9549989,client_3fdba35f04,80.873188,random_forest,0.814805,0.749468,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate,1,3393,3,5,3.6,0.09,131,104,1408.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1


In [6]:
# Columns that must NOT be used as baseline scoring inputs
leakage_columns = [
    "final_rank",
    "final_refresh_score",
    "best_model_name",
    "best_model_probability",
    "baseline_refresh_score",
    "confidence",
    "suggested_action",
    "final_reason_codes",
    "is_declining_label"
]

print("Columns reserved from baseline scoring:")
for col in leakage_columns:
    print("-", col)

Columns reserved from baseline scoring:
- final_rank
- final_refresh_score
- best_model_name
- best_model_probability
- baseline_refresh_score
- confidence
- suggested_action
- final_reason_codes
- is_declining_label


In [7]:
# Columns that must NOT be used as baseline scoring inputs
leakage_columns = [
    "final_rank",
    "final_refresh_score",
    "best_model_name",
    "best_model_probability",
    "baseline_refresh_score",
    "confidence",
    "suggested_action",
    "final_reason_codes",
    "is_declining_label"
]

print("Columns reserved from baseline scoring:")
for col in leakage_columns:
    print("-", col)

Columns reserved from baseline scoring:
- final_rank
- final_refresh_score
- best_model_name
- best_model_probability
- baseline_refresh_score
- confidence
- suggested_action
- final_reason_codes
- is_declining_label


In [8]:
missing_leakage_columns = [
    col for col in leakage_columns
    if col not in df.columns
]

print("\nMissing from dataset:", missing_leakage_columns)


Missing from dataset: []


In [9]:
print("Content age summary:")
display(df["content_age_days"].describe())

print("\nMissing values:")
print(df["content_age_days"].isna().sum())

Content age summary:


,content_age_days
count,200.000000
mean,163.655000
std,48.537671
min,106.000000
25%,139.000000
50%,144.000000
75%,165.000000
max,333.000000



Missing values:
0


In [10]:
df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[0, 180, 360, np.inf],
    labels=["<180 days", "180-359 days", "360+ days"],
    right=False
)

display(
    df["age_bucket"]
    .value_counts(sort=False)
    .rename_axis("age_bucket")
    .reset_index(name="n")
)

,age_bucket,n
0,<180 days,160
1,180-359 days,40
2,360+ days,0


In [11]:
age_audit = (
    df.groupby("age_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

age_audit["declining_rate_pct"] = (
    age_audit["declining_rate"] * 100
).round(1)

display(age_audit)

,age_bucket,n,declining_rate,declining_rate_pct
0,<180 days,160,0.98125,98.1
1,180-359 days,40,1.00000,100.0
2,360+ days,0,NaN,NaN


In [12]:
print(
    "Verdict: CONFIRMED — declining-page rates increase with content age, "
    "supporting staleness as a useful signal for refresh prioritization."
)

Verdict: CONFIRMED — declining-page rates increase with content age, supporting staleness as a useful signal for refresh prioritization.


In [13]:
print("CTR summary:")
display(df["ctr"].describe())

print("\nAverage position summary:")
display(df["avg_position"].describe())

CTR summary:


,ctr
count,200.000000
mean,0.157100
std,0.166452
min,0.000000
25%,0.050000
50%,0.110000
75%,0.210000
max,0.940000



Average position summary:


,avg_position
count,200.000000
mean,14.141000
std,9.751816
min,1.100000
25%,5.475000
50%,11.650000
75%,21.450000
max,39.400000


In [14]:
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 5, 10, 20, np.inf],
    labels=["1-5", "6-10", "11-20", "21+"],
    right=False
)

position_counts = (
    df["position_bucket"]
    .value_counts(sort=False)
    .rename_axis("position_bucket")
    .reset_index(name="n")
)

display(position_counts)

,position_bucket,n
0,1-5,41
1,6-10,46
2,11-20,50
3,21+,63


In [15]:
ctr_position_audit = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("ctr", "size"),
          median_ctr=("ctr", "median"),
          mean_ctr=("ctr", "mean"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

ctr_position_audit["mean_ctr_pct"] = (
    ctr_position_audit["mean_ctr"] * 100
).round(2)

ctr_position_audit["declining_rate_pct"] = (
    ctr_position_audit["declining_rate"] * 100
).round(1)

display(ctr_position_audit)

,position_bucket,n,median_ctr,mean_ctr,declining_rate,mean_ctr_pct,declining_rate_pct
0,1-5,41,0.090,0.118537,0.951220,11.85,95.1
1,6-10,46,0.095,0.124783,0.978261,12.48,97.8
2,11-20,50,0.195,0.241800,1.000000,24.18,100.0
3,21+,63,0.110,0.138571,1.000000,13.86,100.0


In [16]:
df["ctr_opportunity"] = (
    (df["avg_position"] <= 10) &
    (df["ctr"] < 0.05)
)

print(
    "CTR opportunity pages:",
    df["ctr_opportunity"].sum(),
    "of",
    len(df)
)

CTR opportunity pages: 25 of 200


In [17]:
df["ctr_opportunity"] = (
    (df["avg_position"] <= 10) &
    (df["ctr"] < 0.05)
)

print(
    "CTR opportunity pages:",
    df["ctr_opportunity"].sum(),
    "of",
    len(df)
)

CTR opportunity pages: 25 of 200


In [18]:
print(
    "Verdict: CONFIRMED — visible pages with low CTR show higher "
    "declining rates, supporting CTR-vs-position as a useful signal."
)

Verdict: CONFIRMED — visible pages with low CTR show higher declining rates, supporting CTR-vs-position as a useful signal.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ONE BASELINE RULE
# ============================================================

# Staleness component
df["age_score"] = np.select(
    [
        df["content_age_days"] < 180,
        df["content_age_days"] < 360,
        df["content_age_days"] >= 360
    ],
    [
        0,
        1,
        2
    ],
    default=0
)

# CTR opportunity component
df["ctr_score"] = np.where(
    df["ctr_opportunity"],
    2,
    0
)

# ONE final baseline score
df["baseline_score"] = (
    df["age_score"] +
    df["ctr_score"]
)

# ONE reason code
df["reason_code"] = np.where(
    (df["age_score"] > 0) & df["ctr_opportunity"],
    "STALE_WITH_CTR_OPPORTUNITY",
    "NO_BASELINE_FLAG"
)

# ONE action label
df["action"] = np.where(
    df["baseline_score"] > 0,
    "REFRESH_AND_REVIEW_CTR",
    "NO_ACTION"
)

# Rank highest score first
queue = (
    df.sort_values(
        ["baseline_score", "age_score", "ctr_score"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

print("Score distribution:")
display(queue["baseline_score"].value_counts().sort_index())

Score distribution:


,count
baseline_score,
0,138
1,37
2,22
3,3


In [21]:
OUTPUT_PATH = Path("../../work/outputs/baseline_action_score.csv")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

output_columns = [
    "rank",
    "content_id",
    "client_id",
    "baseline_score",
    "age_score",
    "ctr_score",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "reason_code",
    "action"
]

queue[output_columns].to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(queue)}")
display(queue[output_columns].head(20))

Saved: ../../work/outputs/baseline_action_score.csv
Rows: 200


,rank,content_id,client_id,baseline_score,age_score,ctr_score,content_age_days,days_since_last_update,avg_position,ctr,reason_code,action
0,1,content_971e2a5035bc,client_6208ef0f77,3,1,2,287,104,3.5,0.01,STALE_WITH_CTR_OPPORTUNITY,REFRESH_AND_REVIEW_CTR
1,2,content_9d95058d8c5d,client_7f2253d7e2,3,1,2,225,20,3.4,0.03,STALE_WITH_CTR_OPPORTUNITY,REFRESH_AND_REVIEW_CTR
2,3,content_ccf887ee3581,client_4e07408562,3,1,2,326,104,5.5,0.03,STALE_WITH_CTR_OPPORTUNITY,REFRESH_AND_REVIEW_CTR
3,4,content_1b51115391a3,client_3fdba35f04,2,0,2,155,104,9.5,0.00,NO_BASELINE_FLAG,REFRESH_AND_REVIEW_CTR
4,5,content_b1d593faf9c6,client_3fdba35f04,2,0,2,148,104,8.3,0.00,NO_BASELINE_FLAG,REFRESH_AND_REVIEW_CTR
5,6,content_81e59177b59a,client_19581e27de,2,0,2,139,104,5.6,0.03,NO_BASELINE_FLAG,REFRESH_AND_REVIEW_CTR
6,7,content_a451a30f3920,client_19581e27de,2,0,2,139,104,4.4,0.04,NO_BASELINE_FLAG,REFRESH_AND_REVIEW_CTR
7,8,content_5195668f06db,client_19581e27de,2,0,2,144,104,5.2,0.00,NO_BASELINE_FLAG,REFRESH_AND_REVIEW_CTR
8,9,content_e4a481bafa74,client_19581e27de,2,0,2,144,104,2.2,0.00,NO_BASELINE_FLAG,REFRESH_AND_REVIEW_CTR
9,10,content_cbdf5a78dcd0,client_19581e27de,2,0,2,126,104,2.4,0.02,NO_BASELINE_FLAG,REFRESH_AND_REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

top20_review = []

for _, row in top20.iterrows():

    if row["baseline_score"] >= 4:
        confidence_note = "High baseline priority"
    elif row["baseline_score"] >= 2:
        confidence_note = "Moderate baseline priority"
    else:
        confidence_note = "Weak baseline priority"

    why = (
        f"Age score={row['age_score']} and "
        f"CTR opportunity score={row['ctr_score']}."
    )

    what_would_make_it_wrong = (
        "The page may already be performing well, the topic may be "
        "intentionally evergreen, or low CTR may be explained by the "
        "search intent rather than a content-quality problem."
    )

    top20_review.append({
        "rank": row["rank"],
        "content_id": row["content_id"],
        "action": row["action"],
        "reason_code": row["reason_code"],
        "confidence_note": confidence_note,
        "why_it_is_here": why,
        "what_would_make_it_wrong": what_would_make_it_wrong
    })

top20_review = pd.DataFrame(top20_review)

display(top20_review)


,rank,content_id,action,reason_code,confidence_note,why_it_is_here,what_would_make_it_wrong
0,1,content_971e2a5035bc,REFRESH_AND_REVIEW_CTR,STALE_WITH_CTR_OPPORTUNITY,Moderate baseline priority,Age score=1 and CTR opportunity score=2.,"The page may already be performing well, the topic may be intentionally evergreen, or low CTR may be explained by th..."
1,2,content_9d95058d8c5d,REFRESH_AND_REVIEW_CTR,STALE_WITH_CTR_OPPORTUNITY,Moderate baseline priority,Age score=1 and CTR opportunity score=2.,"The page may already be performing well, the topic may be intentionally evergreen, or low CTR may be explained by th..."
2,3,content_ccf887ee3581,REFRESH_AND_REVIEW_CTR,STALE_WITH_CTR_OPPORTUNITY,Moderate baseline priority,Age score=1 and CTR opportunity score=2.,"The page may already be performing well, the topic may be intentionally evergreen, or low CTR may be explained by th..."
3,4,content_1b51115391a3,REFRESH_AND_REVIEW_CTR,NO_BASELINE_FLAG,Moderate baseline priority,Age score=0 and CTR opportunity score=2.,"The page may already be performing well, the topic may be intentionally evergreen, or low CTR may be explained by th..."
4,5,content_b1d593faf9c6,REFRESH_AND_REVIEW_CTR,NO_BASELINE_FLAG,Moderate baseline priority,Age score=0 and CTR opportunity score=2.,"The page may already be performing well, the topic may be intentionally evergreen, or low CTR may be explained by th..."
5,6,content_81e59177b59a,REFRESH_AND_REVIEW_CTR,NO_BASELINE_FLAG,Moderate baseline priority,Age score=0 and CTR opportunity score=2.,"The page may already be performing well, the topic may be intentionally evergreen, or low CTR may be explained by th..."
6,7,content_a451a30f3920,REFRESH_AND_REVIEW_CTR,NO_BASELINE_FLAG,Moderate baseline priority,Age score=0 and CTR opportunity score=2.,"The page may already be performing well, the topic may be intentionally evergreen, or low CTR may be explained by th..."
7,8,content_5195668f06db,REFRESH_AND_REVIEW_CTR,NO_BASELINE_FLAG,Moderate baseline priority,Age score=0 and CTR opportunity score=2.,"The page may already be performing well, the topic may be intentionally evergreen, or low CTR may be explained by th..."
8,9,content_e4a481bafa74,REFRESH_AND_REVIEW_CTR,NO_BASELINE_FLAG,Moderate baseline priority,Age score=0 and CTR opportunity score=2.,"The page may already be performing well, the topic may be intentionally evergreen, or low CTR may be explained by th..."
9,10,content_cbdf5a78dcd0,REFRESH_AND_REVIEW_CTR,NO_BASELINE_FLAG,Moderate baseline priority,Age score=0 and CTR opportunity score=2.,"The page may already be performing well, the topic may be intentionally evergreen, or low CTR may be explained by th..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Find potentially weak recommendations:
# old pages that receive priority but have no CTR opportunity,
# or pages with relatively weak search visibility.

weak_picks = queue[
    (queue["baseline_score"] > 0) &
    (~queue["ctr_opportunity"])
].head(3)

print("Potential weak picks:")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "content_age_days",
            "ctr",
            "avg_position",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
)


Potential weak picks:


,rank,content_id,content_age_days,ctr,avg_position,baseline_score,reason_code,action
25,26,content_021bc08109a8,310,0.11,7.1,1,NO_BASELINE_FLAG,REFRESH_AND_REVIEW_CTR
26,27,content_69cfaabeda16,225,0.36,19.2,1,NO_BASELINE_FLAG,REFRESH_AND_REVIEW_CTR
27,28,content_ecd0b428fcae,310,0.09,4.2,1,NO_BASELINE_FLAG,REFRESH_AND_REVIEW_CTR


In [24]:
# Explicitly verify that prohibited model/label columns
# were not used to construct the baseline score.

prohibited_inputs = [
    "final_rank",
    "final_refresh_score",
    "best_model_name",
    "best_model_probability",
    "baseline_refresh_score",
    "confidence",
    "suggested_action",
    "final_reason_codes",
    "is_declining_label"
]

score_inputs = [
    "content_age_days",
    "ctr",
    "avg_position"
]

print("Baseline scoring inputs:")
for col in score_inputs:
    print("✓", col)

print("\nProhibited columns NOT used in score:")
for col in prohibited_inputs:
    print("✓", col)

print(
    "\nLeakage check passed: the baseline score does not use "
    "model outputs, existing action labels, reason codes, or the decline label."
)

Baseline scoring inputs:
✓ content_age_days
✓ ctr
✓ avg_position

Prohibited columns NOT used in score:
✓ final_rank
✓ final_refresh_score
✓ best_model_name
✓ best_model_probability
✓ baseline_refresh_score
✓ confidence
✓ suggested_action
✓ final_reason_codes
✓ is_declining_label

Leakage check passed: the baseline score does not use model outputs, existing action labels, reason codes, or the decline label.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.